# SOLUTION (R version): Tukey’s Range Test – VeryAnts Sales
## Full R Workflow with TukeyHSD, Comparisons & Simulation


## Flowchart: When and How to Use Tukey’s Range Test
```mermaid
flowchart TD
    A[Run One-way ANOVA with aov()] --> B{Is ANOVA p < 0.05?}
    B -->|Yes| C[Perform TukeyHSD(aov_object)]
    B -->|No| Stop[Stop - No evidence of differences]
    C --> D[Inspect output: diff, lwr, upr, p adj, reject]
    D --> E{p adj < 0.05?}
    E -->|Yes| F[Significant difference between pair]
    E -->|No| G[Not significant after correction]
    F --> H[Report mean diff + CI + effect size]
    G --> H
    H --> I[Audience-aware conclusion]
```
**Key advantage:** Tukey controls family-wise error rate while being more powerful than Bonferroni for all-pairwise comparisons.


## 1. Setup and Data Loading (Solution)


In [ ]:
library(tidyverse)
library(ggplot2)

veryants <- read_csv("veryants.csv")
print(head(veryants, 3))
print(table(veryants$Store))


## 2. Run Tukey’s Range Test (Solution)

**Result:** Only A vs B is significant after Tukey correction.


In [ ]:
anova_model <- aov(Sale ~ Store, data = veryants)
tukey_results <- TukeyHSD(anova_model)
print(tukey_results)

cat("\n=== Only A vs B is significant after correction ===\n")


## 3. Interpretation (Solution)

**Key findings:**
- A vs B: significant (diff ≈ +7.28, p adj < 0.001)
- A vs C and B vs C: not significant after Tukey correction

This is more conservative than the three separate t-tests from the previous exercise (where A vs C had raw p=0.021).


## 4. Significance Flags (Solution)


In [ ]:
a_b_significant <- TRUE
a_c_significant <- FALSE
b_c_significant <- FALSE

cat(sprintf("A vs B significant (Tukey): %s\n", a_b_significant))
cat(sprintf("A vs C significant (Tukey): %s\n", a_c_significant))
cat(sprintf("B vs C significant (Tukey): %s\n", b_c_significant))


## 5. More Practice Answers (Solution)


In [ ]:
# 1. Holm correction
pairwise.t.test(veryants$Sale, veryants$Store, p.adjust.method = "holm")

# 2. Cohen's d for A vs B
a <- veryants$Sale[veryants$Store == "A"]
b <- veryants$Sale[veryants$Store == "B"]
pooled_sd <- sqrt( ((length(a)-1)*var(a) + (length(b)-1)*var(b)) / (length(a) + length(b) - 2) )
d <- (mean(b) - mean(a)) / pooled_sd
cat(sprintf("Cohen's d for A vs B: %.3f (medium effect)\n", d))


## 6. Simulation (Full Working R Version)


In [ ]:
set.seed(42)

true_means <- c(58, 65, 62)   # try c(60,60,60) for null
sigma <- 15
n_per_group <- 150
n_simulations <- 300
alpha <- 0.05

results <- replicate(n_simulations, {
  g1 <- rnorm(n_per_group, true_means[1], sigma)
  g2 <- rnorm(n_per_group, true_means[2], sigma)
  g3 <- rnorm(n_per_group, true_means[3], sigma)
  
  df_sim <- data.frame(
    value = c(g1, g2, g3),
    group = rep(c("A", "B", "C"), each = n_per_group)
  )
  
  anova_sim <- aov(value ~ group, data = df_sim)
  tukey_sim <- TukeyHSD(anova_sim)
  
  any(tukey_sim[[1]][, "p adj"] < alpha)
})

detection_rate <- mean(results)
label <- if (all(true_means == true_means[1])) "Family-wise error rate" else "Power (at least one significant pair)"
cat(sprintf("%s: %.3f\n", label, detection_rate))
cat("Tukey maintains good error control while detecting real differences.\n")


## 7. Example Conclusion & Audience Reporting (R Solution)

### Overall Conclusion
Tukey’s HSD test following a significant ANOVA showed that only Store B had significantly higher average sales than Store A (mean difference ≈ 7.28 USD, p-adj < 0.001). Differences involving Store C were not statistically significant after correction. This is more conservative than running three separate t-tests.

**Recommendation:** Investigate what drives higher sales at Store B.

### Audience-tailored versions

**Executives:**
> "Tukey’s test confirmed that Store B has significantly higher average sales than Store A (about $7 more per sale). We should study what Store B is doing well."

**Technical:**
> "Tukey HSD after significant ANOVA: only A vs B significant (diff=7.28, p-adj<0.001). A vs C and B vs C not significant after FWER correction. More conservative than uncorrected t-tests. Cohen’s d ≈ 0.49 (medium)."

**Non-technical:**
> "We used a careful test (Tukey) that checks all store pairs while avoiding false alarms. It showed that Store B really does have higher sales than Store A. The other comparisons weren’t strong enough to be sure."
